# 03. Model Training & Benchmarking Comparison

This notebook covers:
- Preprocessing with `RobustScaler` and `OneHotEncoder`
- Unsupervised model training across Isolation Forest, Local Outlier Factor (LOF), DBSCAN, and PyOD ECOD
- Standardization of continuous anomaly scores into [0, 1]
- Percentile-based risk thresholding (Top 2% High Risk, Top 5% Suspicious)
- Model detection overlap analysis.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils import load_scored_data

df_scored = load_scored_data()
print(f"Loaded scored dataset: {len(df_scored)} rows.")

## 1. Score Distributions Across Detectors

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(df_scored['score_isolation_forest'], kde=True, ax=axes[0,0], color='blue').set_title('Isolation Forest Score')
sns.histplot(df_scored['score_lof'], kde=True, ax=axes[0,1], color='orange').set_title('Local Outlier Factor (LOF) Score')
sns.histplot(df_scored['score_dbscan'], kde=True, ax=axes[1,0], color='green').set_title('DBSCAN Score')
sns.histplot(df_scored['score_pyod_ecod'], kde=True, ax=axes[1,1], color='purple').set_title('PyOD ECOD Score')
plt.tight_layout()
plt.show()

## 2. Model Pairwise Overlap Jaccard Matrix

In [ ]:
if_set = set(df_scored[df_scored['pred_isolation_forest'] == 1].index)
lof_set = set(df_scored[df_scored['pred_lof'] == 1].index)
dbscan_set = set(df_scored[df_scored['pred_dbscan'] == 1].index)
pyod_set = set(df_scored[df_scored['pred_pyod_ecod'] == 1].index)

def jaccard(s1, s2):
    return len(s1.intersection(s2)) / len(s1.union(s2)) * 100

overlap_df = pd.DataFrame([
    [100.0, jaccard(if_set, lof_set), jaccard(if_set, dbscan_set), jaccard(if_set, pyod_set)],
    [jaccard(lof_set, if_set), 100.0, jaccard(lof_set, dbscan_set), jaccard(lof_set, pyod_set)],
    [jaccard(dbscan_set, if_set), jaccard(dbscan_set, lof_set), 100.0, jaccard(dbscan_set, pyod_set)],
    [jaccard(pyod_set, if_set), jaccard(pyod_set, lof_set), jaccard(pyod_set, dbscan_set), 100.0]
], columns=['Isolation Forest', 'LOF', 'DBSCAN', 'PyOD ECOD'], index=['Isolation Forest', 'LOF', 'DBSCAN', 'PyOD ECOD']).round(1)

overlap_df